In [6]:
import cdflib.xarray
import pandas as pd

### Converting Raw .cdf datasets to .parquet datasets

In [7]:
# For WIND Dataset
ds1 = cdflib.xarray.cdf_to_xarray("../Dataset/raw-dataset/cdf/WIND.cdf", fillval_to_nan=True)
df1 = ds1.to_dataframe()
print(f"No. of rows in WIND Dataset: {len(df1)}")
df1.to_parquet("../Dataset/raw-dataset/parquet/wind.parquet", engine="pyarrow")
print("Successfully converted WIND.cdf to Parquet!")

# For GOES Dataset
ds2 = cdflib.xarray.cdf_to_xarray("../Dataset/raw-dataset/cdf/GOES.cdf", fillval_to_nan=True)
df2 = ds2.to_dataframe()
print(f"No. of rows in GOES Dataset: {len(df2)}")
df2.to_parquet("../Dataset/raw-dataset/parquet/goes.parquet", engine="pyarrow")
print("Successfully converted GOES.cdf to Parquet!")

No. of rows in WIND Dataset: 4733281
Successfully converted WIND.cdf to Parquet!
No. of rows in GOES Dataset: 4727521
Successfully converted GOES.cdf to Parquet!


### Dataset Re-Indexing and Merging

In [9]:
wind_df = pd.read_parquet("../Dataset/raw-dataset/parquet/wind.parquet")
goes_df = pd.read_parquet("../Dataset/raw-dataset/parquet/goes.parquet")

print(f"Total Rows in WIND Dataset: {len(wind_df)}")
print(f"Total Rows in GOES Dataset: {len(goes_df)}")


# Creating a continuous 1-minute time grid for the entire 9 years
full_time_grid = pd.date_range(
    start="2011-01-01 00:00:00", 
    end="2020-01-01 00:00:00", 
    freq="1min", 
    name="Epoch"
)

# Reindexing both DataFrames to the full time grid
wind_df = wind_df.reindex(full_time_grid)
goes_df = goes_df.reindex(full_time_grid)

# Merge side-by-side
merged_df = wind_df.join(goes_df)
print(f"Total Rows in Merged Dataset: {len(merged_df)}")

# Save the merged file
merged_df.to_parquet("../Dataset/cleaned-dataset/merged-dataset.parquet", engine="pyarrow")
print("Merged Dataset saved successfully")


Total Rows in WIND Dataset: 4733281
Total Rows in GOES Dataset: 4727521
Total Rows in Merged Dataset: 4733281
Merged Dataset saved successfully


### Missing Data Analysis

In [11]:

df = pd.read_parquet("../Dataset/cleaned-dataset/merged-dataset.parquet")

print("========= No. of Misssing records in Each Column =========")
print(df.isna().sum())

print("\n\n========= Percent of Misssing records in Each Column =========")
print((df.isna().sum() / len(df) * 100).sort_values(ascending=False))

========= No. of Misssing records in Each Column =========
F                  356373
BX_GSE             356373
BY_GSM             356373
BZ_GSM             356373
flow_speed        1118140
proton_density    1118140
E2W_COR_FLUX      1359975
dtype: int64


========= Percent of Misssing records in Each Column =========
E2W_COR_FLUX      28.732184
flow_speed        23.622937
proton_density    23.622937
BY_GSM             7.529090
BX_GSE             7.529090
F                  7.529090
BZ_GSM             7.529090
dtype: float64


### Gap Analysis

In [12]:

def analyze_gap_lengths(series, name):  
    # Identify contiguous blocks of NaNs
    is_na = series.isna()
    blocks = (~is_na).cumsum()[is_na]
    gap_lengths = blocks.value_counts()
    
    print(f"=== Gap Analysis for {name} ===")
    print(f"Max single gap: {gap_lengths.max() if not gap_lengths.empty else 0} consecutive minutes")
    print(f"Gaps <= 15 mins: {(gap_lengths <= 15).sum()} occurrences")
    print(f"Gaps 16-30 mins: {((gap_lengths > 15) & (gap_lengths <= 30)).sum()} occurrences")
    print(f"Gaps > 30 mins:   {(gap_lengths > 30).sum()} occurrences\n")


for col in df.columns:
    # To skip timestamp column
    if col.lower() =='epoch':
        continue
    
    analyze_gap_lengths(df[col], col)

=== Gap Analysis for F ===
Max single gap: 3311 consecutive minutes
Gaps <= 15 mins: 111287 occurrences
Gaps 16-30 mins: 281 occurrences
Gaps > 30 mins:   1325 occurrences

=== Gap Analysis for BX_GSE ===
Max single gap: 3311 consecutive minutes
Gaps <= 15 mins: 111287 occurrences
Gaps 16-30 mins: 281 occurrences
Gaps > 30 mins:   1325 occurrences

=== Gap Analysis for BY_GSM ===
Max single gap: 3311 consecutive minutes
Gaps <= 15 mins: 111287 occurrences
Gaps 16-30 mins: 281 occurrences
Gaps > 30 mins:   1325 occurrences

=== Gap Analysis for BZ_GSM ===
Max single gap: 3311 consecutive minutes
Gaps <= 15 mins: 111287 occurrences
Gaps 16-30 mins: 281 occurrences
Gaps > 30 mins:   1325 occurrences

=== Gap Analysis for flow_speed ===
Max single gap: 3315 consecutive minutes
Gaps <= 15 mins: 360888 occurrences
Gaps 16-30 mins: 2851 occurrences
Gaps > 30 mins:   2013 occurrences

=== Gap Analysis for proton_density ===
Max single gap: 3315 consecutive minutes
Gaps <= 15 mins: 360888 occur

### Interpolation

In [13]:
print("=== Number of NaNs BEFORE CLEANING ===")
print(df.isna().sum())
print("\n=== Percentage of NaNs BEFORE CLEANING ===")
print((df.isna().sum() / len(df) * 100).sort_values(ascending=False))

# Linear Interpolation for Gap's < 15min
df_clean = df.interpolate(method="time", limit=15)

# Forward-filling medium gaps (<= 30 mins) ONLY for input features
wind_features = ["F", "BX_GSE", "BY_GSM", "BZ_GSM", "flow_speed", "proton_density"]
df_clean[wind_features] = df_clean[wind_features].ffill(limit=30)

print("\n\n=== Number of NaNs AFTER CLEANING ===")
print(df_clean.isna().sum())
print("\n=== Percentage of NaNs AFTER CLEANING ===")
print((df_clean.isna().sum() / len(df) * 100).sort_values(ascending=False))

df_clean.to_parquet("../Dataset/cleaned-dataset/merged-dataset-cleaned.parquet", engine="pyarrow")
print("\nSuccessfully saved cleaned dataset")

=== Number of NaNs BEFORE CLEANING ===
F                  356373
BX_GSE             356373
BY_GSM             356373
BZ_GSM             356373
flow_speed        1118140
proton_density    1118140
E2W_COR_FLUX      1359975
dtype: int64

=== Percentage of NaNs BEFORE CLEANING ===
E2W_COR_FLUX      28.732184
flow_speed        23.622937
proton_density    23.622937
BY_GSM             7.529090
BX_GSE             7.529090
F                  7.529090
BZ_GSM             7.529090
dtype: float64


=== Number of NaNs AFTER CLEANING ===
F                  127812
BX_GSE             127812
BY_GSM             127812
BZ_GSM             127812
flow_speed         132690
proton_density     132690
E2W_COR_FLUX      1060077
dtype: int64

=== Percentage of NaNs AFTER CLEANING ===
E2W_COR_FLUX      22.396241
flow_speed         2.803341
proton_density     2.803341
BY_GSM             2.700283
BX_GSE             2.700283
F                  2.700283
BZ_GSM             2.700283
dtype: float64

Successfully saved cl